# RL-RNN: economic choice in an excitatory-inhibitory actor-critic network

This notebook trains a recurrent neural network with **reinforcement learning** instead of
supervised learning, and reproduces the central results of

> Battista, Padoa-Schioppa & Wang (2026), *A neural circuit framework for economic choice: from building blocks of valuation to compositionality in multitasking*. Neuron

The paper asks a circuit-level question: when a biologically constrained recurrent network
learns economic choices from reward feedback alone, what mechanism does it find? The answer
has two stages. First, values are *computed* in the feedforward pathway: the input weights
come to store the subjective values of the goods and approximate the multiplication of
quantity by probability. Second, values are *compared* inside the recurrent circuit through
a specific excitatory-inhibitory connectivity motif (competitive recurrent inhibition, CRI)
that implements a winner-take-all decision. A network trained on five tasks at once combines
a shared computational core with task-specific modules.

We follow the same logic here with one trained network (the paper pools over 20-50 networks;
where an analysis requires that, we say so and skip it).

**How this maps onto the NeuralRNN layers.** Everything in the previous notebooks was
supervised: a dataset produced batches, an objective compared the readout to a target.
Here the data come from an *environment* that is stepped online, and the loss is a
reinforcement learning loss:

| layer | supervised notebooks | this notebook |
|---|---|---|
| data | `Dataset.sample_batch()` | `neuralrnn.rl.envs` (`make_env`, `SyncVectorEnv`), stepped online |
| model | `NeuralDynamicsModel` + readout | `ActorCriticModel`: any registered RNN core + actor and critic heads |
| loss | `Objective` (CE / MSE) | `PPOLoss` (`neuralrnn.rl.losses`, pluggable via `register_rl_loss`) |
| trainer | `Trainer` | `RLTrainer` (`neuralrnn.rl.trainer`): recurrent rollouts, n-step returns, BPTT |

**Contents**

1. The ECHOICE task battery (the environment)
2. The agent: a Dale-constrained continuous-time RNN with actor and critic heads
3. PPO training, stage 1: the standard task
4. Stage 2: curriculum fine-tuning on all five tasks
5. Choice behavior: the network recovers the subjective values (paper Fig. 2)
6. Single-neuron selectivity (Fig. 3)
7. Low-dimensional population geometry (Fig. 4)
8. Feedforward value computation (Fig. 5)
9. Value comparison in the recurrent circuit: the CRI motif (Fig. 6)
10. Compositionality across tasks (Fig. 7)

Trained checkpoints live in `models/18/` and every training cell is load-first: if a
checkpoint exists it is loaded and training is skipped (delete the corresponding folder to
retrain). Figures are saved to `figs/18/`. Reduced configs are used throughout; the paper's
values are given in the text.

## Setup

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import torch

from neuralrnn import (
    AutoModel, ActorCriticConfig, ActorCriticModel,
    RLTrainer, RLTrainingArguments, SyncVectorEnv, collect_episodes, make_env,
)
from neuralrnn.models.ctrnn import EIRNNConfig
from neuralrnn.rl.envs.echoice import RULE_NAMES
from neuralrnn.analysis.dimensionality import fit_pca

MODEL_DIR = "models/18"
FIG_DIR = "figs/18"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
DEVICE = "cpu"   # the recurrent model is small; CPU beats GPU here (kernel-launch overhead)

# analysis constants
DT = 20.          # ms per simulation step
GOODS = ["A", "B", "C", "D", "E"]
RHO = {"A": 3.0, "B": 2.5, "C": 2.0, "D": 1.5, "E": 1.0}   # ground-truth relative values
print("rules:", RULE_NAMES)

## 1. The ECHOICE task battery

The environment reimplements the ECHOICE battery of the paper. There are five goods, A to E, with fixed relative values

$$\rho_A = 3,\quad \rho_B = 2.5,\quad \rho_C = 2,\quad \rho_D = 1.5,\quad \rho_E = 1,$$

which the network never observes directly: they only enter through the reward it receives.
The value of an offer of good $X$ is

$$\mathrm{OV}_X = \rho_X \cdot q_X \cdot p_X,$$

where $q_X$ is the offered quantity (sampled uniformly from $[0, 10/\rho_X]$, so that all
goods span a comparable value range) and $p_X$ the reward probability. Offers with
$\mathrm{OV}_X < 1$ are redrawn, so every choice is non-trivial. Choosing an offer yields a
terminal reward equal to $\rho_X q_X$ with probability $p_X$, and 0 otherwise.

**Trial structure.** Each trial starts with a fixation epoch (500-1500 ms, jittered so the
network cannot use time as a cue): the agent must hold action 0, and breaking fixation aborts
the trial with reward -1. A rule cue then signals which of the five tasks is in effect, the
offer(s) appear, and after a go cue (fixation off) the agent has up to 1000 ms to respond.
The four available actions are: 0 = hold fixation, 1 = choose good A (or bundle), 2 = choose
good C (or bundle B+C), 3 = choose good E (or bundle D+E).

| rule | name | offers | what it demands |
|---|---|---|---|
| 0 | standard | C vs E, $p=1$ | plain value comparison |
| 1 | risky | C vs E, $p$ varies | multiplication of quantity and probability |
| 2 | sequential | C and E shown one after another | working memory of the first offer |
| 3 | ternary | A vs C vs E | three-way comparison |
| 4 | bundle | B+C vs D+E | summation of values within an offer |

**Observation vector (16 channels, dt = 20 ms).** Channels 0-4 are one-hot rule cues,
channel 5 is the fixation flag, channels 6-10 the five quantities (divided by 10), and
channels 11-15 the five probabilities. Every channel carries a baseline $u_0 = 0.2$ plus
Gaussian noise with $\sigma_{in} = 0.01$.

The cell below runs one trial per rule with a scripted perfect policy (hold fixation until
the go cue, then pick the offer with the higher expected value), so we can look at what the
network actually sees.

In [ ]:
CHANNELS = ["rule: standard", "rule: risky", "rule: seq.", "rule: ternary", "rule: bundle",
            "fixation", "q_A", "q_B", "q_C", "q_D", "q_E",
            "p_A", "p_B", "p_C", "p_D", "p_E"]

def run_scripted_trial(rule, seed=0):
    '''One trial under a perfect hand-coded policy, for inspection.'''
    env = make_env("echochoice", rules=(rule,), seed=seed)()
    obs, info = env.reset()
    traj, done = [], False
    while not done:
        if env.t < env.t_gon:
            a = 0                                    # hold fixation
        elif rule in (0, 1, 2):                      # binary C vs E
            a = 2 if env.evc >= env.eve else 3
        elif rule == 3:                              # ternary A vs C vs E
            a = 1 + int(np.argmax([env.eva, env.evc, env.eve]))
        else:                                        # bundle B+C vs D+E
            a = 2 if env.evc + env.evb >= env.eve + env.evd else 3
        obs, r, term, trunc, info = env.step(a)
        traj.append(obs.copy())
        done = term or trunc
    return np.asarray(traj), info["episode"]

EPOCH_COLORS = {"fix": "#f0f0f0", "rule": "#fdebcf", "offer": "#dbe9f6", "dec": "#e2f0d9"}

fig, axes = plt.subplots(5, 1, figsize=(9, 12), sharex=False)
for ax, rule in zip(axes, range(5)):
    traj, ep = run_scripted_trial(rule)
    t = np.arange(len(traj)) * DT
    t_fix_end = ep["t_fix"]
    t_offer_on = ep["t_fix"] + ep["t_rule"]
    t_go = ep["t_go"]
    ax.axvspan(0, t_fix_end, color=EPOCH_COLORS["fix"], zorder=0)
    ax.axvspan(t_fix_end, t_offer_on, color=EPOCH_COLORS["rule"], zorder=0)
    ax.axvspan(t_offer_on, t_go, color=EPOCH_COLORS["offer"], zorder=0)
    ax.axvspan(t_go, t[-1], color=EPOCH_COLORS["dec"], zorder=0)
    for ch in range(5, 16):
        if np.ptp(traj[:, ch]) > 1e-6 or ch == 5:
            ax.plot(t, traj[:, ch], label=CHANNELS[ch], lw=1.2)
    ax.set_ylim(0, None)
    ax.set_ylabel("input", fontsize=10)
    ax.set_title(f"rule {rule} ({RULE_NAMES[rule]}): scripted choice earns reward "
                 f"{ep['r']:.2f}", fontsize=10)
    ax.legend(fontsize=9, ncol=2, frameon=False, bbox_to_anchor=(1.02, 1.02))
    if rule == 0:
        for x, lab in [(t_fix_end / 2, "fixation"), ((t_fix_end + t_offer_on) / 2, "rule cue"),
                       ((t_offer_on + t_go) / 2, "offer"), ((t_go + t[-1]) / 2, "choice")]:
            ax.text(x, ax.get_ylim()[1] * 0.95, lab, ha="center", fontsize=8, style="italic")
axes[-1].set_xlabel("time from trial start (ms)")
fig.suptitle("ECHOICE observations: one scripted perfect-choice trial per task", y=0.995)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "env_trials.png"), dpi=200)
plt.show()

## 2. The agent: a Dale-constrained continuous-time RNN with actor and critic heads

The network is a continuous-time vanilla RNN (no gating) whose firing-rate vector
$\mathbf{r}(t)$ follows

$$\tau \frac{d\mathbf{r}}{dt} = -\mathbf{r} + \left[ W_{rec}\, \mathbf{r}(t) +
W_{in}\, \mathbf{u}(t) + \mathbf{b} + \sqrt{2\tau}\,\sigma_{rec}\,\boldsymbol{\xi}(t) \right]_+,$$

integrated with Euler steps of $\delta t = 20$ ms and $\tau = 100$ ms, i.e.
$\alpha = \delta t / \tau = 0.2$. $[\cdot]_+$ is a ReLU, so rates stay non-negative.
$N = 256$ units, 80% excitatory and 20% inhibitory, and Dale's law is enforced by storing
$|W_{rec}|$ and applying the fixed column signs on the fly
($W_{rec}^{\mathrm{eff}} = |W_{rec}|\,\mathrm{diag}(\pm 1)$). Recurrent weights are
initialized from a Gamma(4, 4) distribution, rescaled to spectral radius 1.5, with no
self-connections; input and readout weights are non-negative (long-range projections are
excitatory), which is re-imposed after every optimizer step by `project_constraints()`.
The recurrent noise uses the paper's discretization-consistent scaling
$\sqrt{2\sigma^2/\alpha}$ (`noise_scaling="sqrt_2_over_alpha"`).

Two linear readouts sit on the *same* excitatory population:

* the **actor** produces logits over the four actions,
  $\pi_\theta(a|s) = \mathrm{softmax}(\tfrac{4}{7.5} W_{actor} \mathbf{r}_E)$, and
* the **critic** produces a scalar value estimate $V_\theta(s) = W_{critic}\,\mathbf{r}_E$,
  the network's running prediction of discounted future reward.

Sharing one recurrent core between actor and critic is a deliberate design choice of the
paper: value-related areas are causally involved in choice, so separate actor/critic modules
would be biologically implausible.

In [ ]:
N_UNITS = 256

def build_agent(seed=0):
    torch.manual_seed(seed)
    core_cfg = EIRNNConfig(
        input_dim=16, latent_dim=N_UNITS, output_dim=4, dt=20., tau=100.,
        activation="relu", sigma_rec=0.15, noise_scaling="sqrt_2_over_alpha",
        init_method="gamma", spectral_radius=1.5, no_self_connections=True,
    ).to_dict()
    cfg = ActorCriticConfig(core_config=core_cfg, action_dim=4,
                            action_type="discrete", policy_logit_scale=4. / 7.5,
                            actor_positive=True, critic_positive=True,
                            core_input_positive=True)
    return ActorCriticModel(cfg)

agent = build_agent()
print(f"core: {type(agent.core).__name__}  (E units: {agent.core.e_size}, "
      f"I units: {agent.core.i_size})")
print(f"trainable parameters: {sum(p.numel() for p in agent.parameters()):,}")

# one stochastic step, as a sanity check
envs = SyncVectorEnv([make_env("echochoice", rules=(0,), seed=0)])
obs, _ = envs.reset()
z = agent.init_state(1)
a, logp, ent, v, z = agent.step(torch.as_tensor(obs, dtype=torch.float32), z)
print(f"sampled action {a.item()}, log-prob {logp.item():.3f}, value {v.item():.3f}")
envs.close()

## 3. PPO training, stage 1: the standard task

The agent is trained with proximal policy optimization (PPO). Experience is collected by
running 64 copies of the environment in parallel for 256 steps each; the policy is then
updated by gradient ascent on

$$\mathcal{L}(\theta) = \mathbb{E}\left[ \mathcal{L}^{PPO}_t(\theta)
- c_1 \mathcal{L}^{VF}_t(\theta) + c_2\, \mathcal{S}[\pi_\theta](s_t) \right],
\qquad c_1 = 0.5,\; c_2 = 0.01,$$

where the clipped surrogate keeps policy updates small,

$$\mathcal{L}^{PPO}_t(\theta) = -\min\left( \rho_t(\theta) A_t,\;
\mathrm{clip}(\rho_t(\theta), 1-\varepsilon, 1+\varepsilon) A_t \right),
\qquad \varepsilon = 0.1,$$

$\rho_t(\theta) = \pi_\theta(a_t|s_t) / \pi_{\theta_{old}}(a_t|s_t)$, and the advantage is
the reward prediction error against the critic, $A_t = R_t - V_\theta(s_t)$, with the
n-step bootstrapped return

$$R_t = \sum_{k=0}^{T-t-1} \gamma^k r_{t+k} + \gamma^{T-t} V_\theta(s_T), \qquad \gamma = 0.99.$$

The critic is trained on $\mathcal{L}^{VF}_t = \tfrac{1}{2}(V_\theta(s_t) - R_t)^2$ and the
entropy bonus $\mathcal{S}$ keeps the policy exploring. Gradients flow through time over the
256-step chunks; Adam with learning rate $10^{-3}$ and gradient-norm clipping at 1.0.

Training is load-first: the checkpoints shipped in `models/18/stage1` are used if present.
Watch three metrics during training: the *hold rate* (fraction of pre-go steps with the
fixation held), *PD* (percentage of trials with a decision), and *PC* (percentage of decided
trials that pick the higher-value offer). They produce the paper's characteristic
three-phase curve: first the network learns to hold fixation (hold rate climbs while PD
stays at 0), then it learns to answer at all (PD jumps to ~100%, choices still random),
and only then does the value-guided policy emerge (PC rises past 90%).

In [ ]:
STAGE1 = os.path.join(MODEL_DIR, "stage1")
NUM_ENVS, NUM_STEPS = 32, 256

def make_eval_fn(eval_envs, n_episodes=100):
    '''Metrics tracked during training: fixation holding, decisions, correct choices.'''
    def eval_fn(agent):
        eps = collect_episodes(agent, eval_envs, n_episodes, device=DEVICE)
        hold = [float((ep["actions"][:int(ep["info"]["t_go"] / DT)] == 0).mean())
                for ep in eps]
        pd = [ep["info"]["n_dec"] for ep in eps]
        pc = [ep["info"]["n_corr"] for ep in eps if ep["info"]["n_dec"] == 1.]
        return {"hold_rate": float(np.mean(hold)),
                "pd": float(np.mean(pd)) * 100,
                "pc": float(np.mean(pc)) * 100 if pc else 0.0}
    return eval_fn

if os.path.exists(os.path.join(STAGE1, "best", "config.json")):
    print("checkpoint found; loading (delete models/18/stage1 to retrain)")
    agent = AutoModel.from_pretrained(os.path.join(STAGE1, "best"))
else:
    agent = build_agent()
    envs = SyncVectorEnv([make_env("echochoice", rules=(0,), seed=100 + i)
                          for i in range(NUM_ENVS)])
    eval_envs = SyncVectorEnv([make_env("echochoice", rules=(0,), seed=10_000 + i)
                               for i in range(16)])
    args = RLTrainingArguments(
        total_timesteps=15_000_000, num_envs=NUM_ENVS, num_steps=NUM_STEPS,
        estimator="nstep", gamma=0.99, lr=1e-3, seed=0, device=DEVICE,
        output_dir=STAGE1, eval_every=40, save_every=400,
        early_stop_metric="pc", early_stop_threshold=90.0, progress_bar=True)
    RLTrainer(agent, envs, args, eval_fn=make_eval_fn(eval_envs)).train()
    envs.close(); eval_envs.close()

### Learning curves

The three phases are easy to read off the metrics: the hold rate saturates within the first
million steps, PD follows once the network dares to answer, and PC climbs last as the value
policy crystallizes.

In [ ]:
def plot_eval_histories(stage_dirs, out="learning_curves.png"):
    fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharex=True)
    step0 = 0.
    for stage_dir in stage_dirs:
        path = os.path.join(stage_dir, "eval_history.json")
        if not os.path.exists(path):
            print("skip (no eval_history.json):", stage_dir)
            continue
        with open(path) as f:
            hist = json.load(f)
        with open(os.path.join(stage_dir, "rl_training_args.json")) as f:
            a = json.load(f)
        xs = np.array([m["update"] for m in hist]) * a["num_envs"] * a["num_steps"] / 1e6 + step0
        step0 = xs[-1]
        name = {"stage1": "stage 1: standard", "stage_risky": "stage 1b: risky",
                "stage2": "stage 2: all five tasks"}.get(os.path.basename(stage_dir.rstrip("/\\")),
                                                         os.path.basename(stage_dir))
        for ax, key in zip(axes, ("hold_rate", "pd", "pc")):
            ax.plot(xs, [m[key] for m in hist], marker=".", ms=3, lw=1, label=name)
        axes[0].axvline(step0, color="gray", ls=":", lw=1)
    axes[1].axhline(99, color="r", ls="--", lw=1, label="criterion")
    axes[2].axhline(90, color="r", ls="--", lw=1, label="criterion")
    for ax, t, ylab in zip(axes, ("Fixation holding", "Trials with a decision",
                                  "Correct choices"),
                           ("hold rate", "PD (%)", "PC (%)")):
        ax.set_title(t, fontsize=11)
        ax.set_ylabel(ylab)
        ax.set_xlabel("environment steps (millions)")
        ax.grid(alpha=0.3)
    axes[0].set_ylim(-0.05, 1.05)
    axes[2].legend(fontsize=8, loc="lower right")
    fig.suptitle("Three-phase learning: hold fixation, then answer, then choose by value",
                 fontsize=12)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, out), dpi=200)
    plt.show()

plot_eval_histories([STAGE1])

## 4. Stage 2: curriculum fine-tuning on all five tasks

The paper trains its multitask networks directly on all five tasks, but its curriculum
experiments (its Fig. 7F) show that pre-training on simpler tasks substantially accelerates
the harder ones. We use that observation here: starting from the stage-1 agent, we first
fine-tune briefly on the risky task (this adds probabilistic outcomes on top of the learned
fixation and comparison skills), then continue on all five tasks sampled uniformly. Both
phases are load-first. The per-rule PC traces below show that the sequential task (working
memory) is learned slowest, matching the paper's ordering of task difficulty.

In [ ]:
STAGE_RISKY = os.path.join(MODEL_DIR, "stage_risky")
STAGE2 = os.path.join(MODEL_DIR, "stage2")
ALL_RULES = (0, 1, 2, 3, 4)

def make_multitask_eval_fn(eval_envs, n_episodes=200):
    base = make_eval_fn(eval_envs, n_episodes)
    def eval_fn(agent):
        eps = collect_episodes(agent, eval_envs, n_episodes, device=DEVICE)
        hold = [float((ep["actions"][:int(ep["info"]["t_go"] / DT)] == 0).mean())
                for ep in eps]
        pd = [ep["info"]["n_dec"] for ep in eps]
        pc = [ep["info"]["n_corr"] for ep in eps if ep["info"]["n_dec"] == 1.]
        out = {"hold_rate": float(np.mean(hold)),
               "pd": float(np.mean(pd)) * 100,
               "pc": float(np.mean(pc)) * 100 if pc else 0.0}
        for rule in ALL_RULES:
            pcr = [ep["info"]["n_corr"] for ep in eps
                   if ep["info"]["rule"] == rule and ep["info"]["n_dec"] == 1.]
            out[f"pc_r{rule}"] = float(np.mean(pcr)) * 100 if pcr else 0.0
        return out
    return eval_fn

def train_stage(init_from, rules, out_dir, total_steps, lr, seed):
    agent = AutoModel.from_pretrained(init_from)
    envs = SyncVectorEnv([make_env("echochoice", rules=rules, seed=seed * 1000 + i)
                          for i in range(NUM_ENVS)])
    eval_envs = SyncVectorEnv([make_env("echochoice", rules=rules,
                                        seed=seed * 1000 + 50_000 + i)
                               for i in range(16)])
    args = RLTrainingArguments(
        total_timesteps=total_steps, num_envs=NUM_ENVS, num_steps=NUM_STEPS,
        estimator="nstep", gamma=0.99, lr=lr, seed=seed, device=DEVICE,
        output_dir=out_dir, eval_every=40, save_every=400,
        early_stop_metric="pc", early_stop_threshold=90.0, progress_bar=True)
    RLTrainer(agent, envs, args, eval_fn=make_multitask_eval_fn(eval_envs)).train()
    envs.close(); eval_envs.close()

if os.path.exists(os.path.join(STAGE_RISKY, "best", "config.json")):
    print("stage_risky checkpoint found; loading")
else:
    train_stage(os.path.join(STAGE1, "best"), (1,), STAGE_RISKY, 10_000_000, 5e-4, seed=11)

if os.path.exists(os.path.join(STAGE2, "best", "config.json")):
    print("stage2 checkpoint found; loading")
    agent_mt = AutoModel.from_pretrained(os.path.join(STAGE2, "best"))
else:
    train_stage(os.path.join(STAGE_RISKY, "best"), ALL_RULES, STAGE2, 40_000_000, 5e-4,
                seed=12)
    agent_mt = AutoModel.from_pretrained(os.path.join(STAGE2, "best"))

agent_mt.eval()   # analyses below run without recurrent noise
plot_eval_histories([STAGE1, STAGE_RISKY, STAGE2], out="learning_curves.png")

# per-rule PC over stage 2
with open(os.path.join(STAGE2, "eval_history.json")) as f:
    hist2 = json.load(f)
with open(os.path.join(STAGE2, "rl_training_args.json")) as f:
    a2 = json.load(f)
xs = np.array([m["update"] for m in hist2]) * a2["num_envs"] * a2["num_steps"] / 1e6
fig, ax = plt.subplots(figsize=(7, 4))
for rule in ALL_RULES:
    ax.plot(xs, [m[f"pc_r{rule}"] for m in hist2], marker=".", ms=3, lw=1,
            label=RULE_NAMES[rule])
ax.axhline(90, color="r", ls="--", lw=1, label="paper criterion")
ax.set_xlabel("stage-2 environment steps (millions)")
ax.set_ylabel("PC (%)")
ax.set_title("Per-task choice accuracy during multitask fine-tuning")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "per_rule_pc.png"), dpi=200)
plt.show()

## 5. Choice behavior: the network recovers the subjective values

Did the network really learn the hidden values $\rho_X$, or some other heuristic? The paper
answers this with the same logistic regression used for monkey behavior. On the standard
task the model is

$$P(\mathrm{choose\;C}) = \sigma\!\left( \hat{\eta}\, \ln \frac{\hat{\rho}_C\, q_C}{q_E} \right),$$

with two free parameters: choice consistency $\hat\eta$ (the steepness of the psychometric
curve) and the inferred relative value $\hat\rho_C$ (with $\hat\rho_E \equiv 1$ as the
reference). On the risky task a risk-attitude parameter $\hat\gamma$ enters:

$$P(\mathrm{choose\;C}) = \sigma\!\left( \hat{\eta}\, \ln
\frac{\hat{\rho}_C\, q_C\, p_C^{\hat\gamma}}{q_E\, p_E^{\hat\gamma}} \right),$$

so $\hat\gamma > 1$ means risk aversion and $\hat\gamma < 1$ risk seeking. We fit the free
parameters by maximum likelihood on 2000 collected trials per task. If the network learned
the value structure of the environment, we should find $\hat\rho_C \approx 2$ (the ground
truth) and $\hat\gamma \approx 1$, since nothing in training rewards risk aversion or
seeking.

In [ ]:
from scipy.optimize import minimize

def collect_choices(agent, rule, n_episodes=2000, seed=20_000):
    '''Choices and offer parameters on decided trials of one rule.'''
    envs = SyncVectorEnv([make_env("echochoice", rules=(rule,), seed=seed + i)
                          for i in range(8)])
    eps = collect_episodes(agent, envs, n_episodes, device=DEVICE)
    envs.close()
    trials = []
    for ep in eps:
        info = ep["info"]
        if info["n_dec"] != 1. or info["choice"] not in (2, 3):
            continue
        trials.append(dict(nc=info["nc"], ne=info["ne"], pc=info["pc"], pe=info["pe"],
                           chose_c=1.0 if info["choice"] == 2 else 0.0))
    return trials

def fit_binary_logistic(trials, risky):
    '''MLE fit of the paper's Eq. 10 (risky=False) or Eq. 11 (risky=True).'''
    q_c = np.array([t["nc"] for t in trials]); q_e = np.array([t["ne"] for t in trials])
    p_c = np.array([t["pc"] for t in trials]); p_e = np.array([t["pe"] for t in trials])
    y = np.array([t["chose_c"] for t in trials])
    def nll(params):
        eta, log_rho = params[0], params[1]
        gam = params[2] if risky else 1.0
        x = eta * (log_rho + np.log(q_c / q_e) + gam * np.log(p_c / p_e))
        return -np.sum(y * x - np.logaddexp(0., x))
    x0 = [3.0, np.log(2.0)] + ([1.0] if risky else [])
    res = minimize(nll, x0=x0, method="Nelder-Mead",
                   options={"maxiter": 4000, "xatol": 1e-6, "fatol": 1e-8})
    eta, rho_c = res.x[0], np.exp(res.x[1])
    gam = res.x[2] if risky else 1.0
    def predict(xg):  # psychometric curve over the value-ratio axis
        return 1 / (1 + np.exp(-(eta * (np.log(rho_c) + xg))))
    return {"eta": eta, "rho_c": rho_c, "gamma": gam, "n": len(y), "predict": predict}

trials_std = collect_choices(agent_mt, 0)
trials_risk = collect_choices(agent_mt, 1)
fit_std = fit_binary_logistic(trials_std, risky=False)
fit_risk = fit_binary_logistic(trials_risk, risky=True)
print(f"standard: eta = {fit_std['eta']:.2f}, rho_C = {fit_std['rho_c']:.2f} "
      f"(truth 2.0), n = {fit_std['n']}")
print(f"risky   : eta = {fit_risk['eta']:.2f}, rho_C = {fit_risk['rho_c']:.2f}, "
      f"gamma = {fit_risk['gamma']:.2f} (truth: rho 2.0, gamma 1.0), n = {fit_risk['n']}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, trials, fit, risky, name in [
        (axes[0], trials_std, fit_std, False, "standard"),
        (axes[1], trials_risk, fit_risk, True, "risky")]:
    q_c = np.array([t["nc"] for t in trials]); q_e = np.array([t["ne"] for t in trials])
    p_c = np.array([t["pc"] for t in trials]); p_e = np.array([t["pe"] for t in trials])
    y = np.array([t["chose_c"] for t in trials])
    x = np.log(q_c / q_e) + fit["gamma"] * np.log(p_c / p_e)
    xg = np.linspace(x.min(), x.max(), 200)
    bins = np.quantile(x, np.linspace(0, 1, 11))
    centers = [x[(x >= lo) & (x < hi)].mean() for lo, hi in zip(bins[:-1], bins[1:])
               if ((x >= lo) & (x < hi)).sum() > 0]
    fracs = [y[(x >= lo) & (x < hi)].mean() for lo, hi in zip(bins[:-1], bins[1:])
             if ((x >= lo) & (x < hi)).sum() > 0]
    ax.scatter(centers, fracs, c="k", zorder=3, label="network choices (binned)")
    ax.plot(xg, fit["predict"](xg), "r-", lw=1.5,
            label=(rf"fit: $\hat{{\rho}}_C$={fit['rho_c']:.2f}, $\hat{{\eta}}$={fit['eta']:.1f}"
                   + (rf", $\hat{{\gamma}}$={fit['gamma']:.2f}" if risky else "")))
    ax.axhline(0.5, color="gray", ls=":", lw=1)
    ax.axvline(0, color="gray", ls=":", lw=1)
    ax.set_xlabel(r"$\ln(q_C / q_E) + \hat\gamma \ln(p_C / p_E)$"
                  if risky else r"$\ln(q_C / q_E)$")
    ax.set_ylabel("P(choose C)")
    ax.set_title(f"{name} task (n = {fit['n']})")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

ax = axes[2]
labels = ["standard", "risky"]
inferred = [fit_std["rho_c"], fit_risk["rho_c"]]
ax.bar(np.arange(2) - 0.15, inferred, width=0.3, color="#c44e52",
       label=r"inferred $\hat{\rho}_C$")
ax.bar(np.arange(2) + 0.15, [RHO["C"]] * 2, width=0.3, color="#8c8c8c",
       label=r"ground truth $\rho_C = 2$")
for i, v in enumerate(inferred):
    ax.text(i - 0.15, v + 0.04, f"{v:.2f}", ha="center", fontsize=9)
ax.set_xticks(np.arange(2)); ax.set_xticklabels(labels)
ax.set_ylabel("relative value of good C")
ax.set_title("The network recovers the hidden value")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")
ax.set_ylim(0, 2.6)
fig.suptitle("Choice behavior", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "choice_behavior.png"), dpi=200)
plt.show()

The psychometric curves are steep sigmoids: choices are ordered by the computed offer
value. The fit recovers $\hat\rho_C \approx 2$, the ground-truth relative value that only
ever reached the network through noisy reward feedback, and $\hat\gamma \approx 1$: the
network is close to risk neutral, as in the paper (whose 20 networks showed mild
idiosyncratic biases around 1). The behavioral strategy is genuinely value-based.

**Reaction times.** If the network commits to a choice while the offer is still on (we test
this directly on the policy outputs in section 9), then after the go cue it only has to
*execute* the decision, and reaction times should not grow with decision difficulty. The
paper reports exactly this independence (its Fig. S10A).

In [ ]:
# reaction time (go cue -> choice) vs decision difficulty, risky task
envs = SyncVectorEnv([make_env("echochoice", rules=(1,), seed=25_000 + i)
                      for i in range(8)])
eps_rt = collect_episodes(agent_mt, envs, 1500, device=DEVICE)
envs.close()
rts = np.array([ep["info"]["RT"] for ep in eps_rt if ep["info"]["n_dec"] == 1.])
diffs = np.array([abs(ep["info"]["evc"] - ep["info"]["eve"]) for ep in eps_rt
                  if ep["info"]["n_dec"] == 1.])
slope, intercept = np.polyfit(diffs, rts, 1)
r_rt = np.corrcoef(diffs, rts)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(rts, bins=30, color="#4c72b0")
axes[0].set_xlabel("reaction time (ms)")
axes[0].set_ylabel("trials")
axes[0].set_title(f"RT distribution (median {np.median(rts):.0f} ms)")
axes[0].grid(alpha=0.3, axis="y")
axes[1].scatter(diffs, rts, s=4, alpha=0.25, c="#4c72b0")
xg = np.linspace(diffs.min(), diffs.max(), 50)
axes[1].plot(xg, intercept + slope * xg, "r-", lw=1.5,
             label=f"linear fit (r = {r_rt:.2f})")
axes[1].set_xlabel(r"decision difficulty  $|\mathrm{OV}_C - \mathrm{OV}_E|$")
axes[1].set_ylabel("reaction time (ms)")
axes[1].set_title("RT does not depend on difficulty")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
fig.suptitle("Early commitment: behavioral signature (paper Fig. S10A)", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "reaction_times.png"), dpi=200)
plt.show()

## 6. Single-neuron selectivity: the OFC cell types emerge

Electrophysiology in primate orbitofrontal cortex found three functional cell types during
economic choice: **offer value** neurons (encoding the value of one offer), **chosen value**
neurons (encoding the value of whatever was chosen), and **choice** neurons (encoding which
good was chosen). We ask whether the trained network develops the same types.

For each unit we regress its mean activity during the offer epoch on candidate variables,
each with a separate simple regression $r = \beta_0 + \beta X + \varepsilon$:

* offer values $\mathrm{OV}_C = \rho_C q_C p_C$ and $\mathrm{OV}_E$ (the env reports these
  as `evc`, `eve`),
* chosen value $\mathrm{CV}$ (value of the offer the network actually picked),
* choice $\mathrm{CH}$ (1 if C was chosen, 0 if E).

A unit is assigned to the variable that explains it best, provided the fit clears
$R^2 > 0.5$ (the strict criterion; the paper's time-resolved analysis below uses the
standard $p < 0.05$, $R^2 \geq 0.005$). We run this on the binary tasks (standard and
risky pooled, as in the paper).

In [ ]:
from scipy.stats import linregress

# one multitask dataset, reused by sections 6, 7 and 10
envs = SyncVectorEnv([make_env("echochoice", seed=30_000 + i) for i in range(8)])
episodes = collect_episodes(agent_mt, envs, 1200, device=DEVICE)
envs.close()
print(f"collected {len(episodes)} episodes across all five tasks")

def offer_window(info, T=None):
    '''Step indices [t0, t1) of the offer epoch (offer onset to go cue).

    Episodes aborted before the go cue (fixation breaks) end early, so the
    nominal window can run past the end of the recording; pass
    T=len(ep["states"]) to clamp it to what the episode actually contains.
    '''
    t0 = int((info["t_fix"] + info["t_rule"]) / DT)
    t1 = int(info["t_go"] / DT)
    if T is not None:
        t0 = min(t0, max(T - 1, 0))
        t1 = min(t1, T)
    return t0, max(t1, t0 + 1)

def trial_variables(eps, rules=(0, 1)):
    '''Offer-epoch mean activity + decision variables for decided binary trials.'''
    feats, evc, eve, choice = [], [], [], []
    for ep in eps:
        info = ep["info"]
        if info["rule"] not in rules or info["n_dec"] != 1.:
            continue
        if info["choice"] not in (2, 3):
            continue
        t0, t1 = offer_window(info, len(ep["states"]))
        feats.append(ep["states"][t0:t1].mean(axis=0))
        evc.append(info["evc"]); eve.append(info["eve"])
        choice.append(1.0 if info["choice"] == 2 else 0.0)
    H = np.stack(feats)
    evc, eve, choice = np.asarray(evc), np.asarray(eve), np.asarray(choice)
    cv = np.where(choice == 1, evc, eve)
    return H, {"OVC": evc, "OVE": eve, "CV": cv, "CH": choice}

H_bin, V_bin = trial_variables(episodes)
print(f"{H_bin.shape[0]} decided standard/risky trials")

# variable correlation matrix (paper Fig. 3A)
import pandas as pd
Vmat = pd.DataFrame({"offer value C": V_bin["OVC"], "offer value E": V_bin["OVE"],
                     "chosen value": V_bin["CV"], "choice": V_bin["CH"]})
fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(Vmat.corr(), vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(Vmat.columns, rotation=30, ha="right", fontsize=9)
ax.set_yticklabels(Vmat.columns, fontsize=9)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{Vmat.corr().iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="Pearson r", shrink=0.8)
ax.set_title("Decision variables are correlated (Fig. 3A)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "variable_correlations.png"), dpi=200)
plt.show()

The correlation matrix already shows the task structure: chosen value correlates with
both offer values (it is one of them on every trial), and choice correlates with the offer
values because larger $\mathrm{OV}_C$ makes choosing C more likely. Any neural selectivity
has to be interpreted against this background.

Now the tuning proper: example units, the fraction of tuned units in the E and I
populations, and the *sign* of the tuning. Both positive and negative slopes appear in the
paper in both populations; the negative ones are the non-trivial part, because all external
input to the network is excitatory, so negative tuning can only be created by the learned
recurrent inhibition. They turn out to be causal for the decision (section 9).

In [ ]:
R2_THRESHOLD = 0.5
e_size = agent_mt.core.e_size
order = ["OVC", "OVE", "CV", "CH"]
pretty = {"OVC": "offer value C", "OVE": "offer value E",
          "CV": "chosen value", "CH": "choice"}

def regress_units(H, V):
    '''Per unit: R^2 and slope for each candidate variable (offer-epoch means).'''
    r2 = {k: np.array([linregress(x, H[:, u]).rvalue ** 2 for u in range(H.shape[1])])
          for k, x in V.items()}
    sl = {k: np.array([linregress(x, H[:, u]).slope for u in range(H.shape[1])])
          for k, x in V.items()}
    return r2, sl

r2_bin, sl_bin = regress_units(H_bin, V_bin)
best_var = np.array([max(order, key=lambda k: r2_bin[k][u]) for u in range(H_bin.shape[1])],
                    dtype=object)
best_r2 = np.array([r2_bin[best_var[u]][u] for u in range(H_bin.shape[1])])
tuned = best_r2 > R2_THRESHOLD

# --- example tuning curves (paper Fig. 3B) ---
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, var in zip(axes, ("OVE", "CV", "CH")):
    u = int(np.argmax(r2_bin[var]))
    x = V_bin[var]; y = H_bin[:, u]
    ax.scatter(x, y, s=8, alpha=0.3, c="#4c72b0")
    xg = np.linspace(x.min(), x.max(), 50)
    r = linregress(x, y)
    ax.plot(xg, r.intercept + r.slope * xg, "r-", lw=1.5)
    kind = "E" if u < e_size else "I"
    ax.set_title(f"{pretty[var]} unit #{u} ({kind}), $R^2$ = {r.rvalue**2:.2f}")
    ax.set_xlabel(pretty[var]); ax.set_ylabel("mean rate (offer epoch)")
    ax.grid(alpha=0.3)
fig.suptitle("Example single-unit tuning (paper Fig. 3B)", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "tuning_examples.png"), dpi=200)
plt.show()

# --- tuned fractions by population + slope signs (paper Figs. 3C summary / 3F) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
x = np.arange(len(order))
fe = [float((tuned[:e_size] & (best_var[:e_size] == c)).mean()) for c in order]
fi = [float((tuned[e_size:] & (best_var[e_size:] == c)).mean()) for c in order]
ax.bar(x - 0.17, fe, width=0.34, color="#4c72b0", label=f"E units (n = {e_size})")
ax.bar(x + 0.17, fi, width=0.34, color="#c44e52",
       label=f"I units (n = {H_bin.shape[1] - e_size})")
ax.set_xticks(x); ax.set_xticklabels([pretty[c] for c in order], rotation=15)
ax.set_ylabel(f"fraction of units ($R^2$ > {R2_THRESHOLD})")
ax.set_title("Tuned fractions (Fig. 3 summary)")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")

ax = axes[1]
pos_frac, neg_frac = {}, {}
for pop, sl in (("E", slice(0, e_size)), ("I", slice(e_size, None))):
    sel = tuned[sl]
    slopes = np.array([sl_bin[best_var[u]][u]
                       for u in np.arange(H_bin.shape[1])[sl][sel]])
    pos_frac[pop] = float((slopes > 0).mean()) if len(slopes) else 0.
    neg_frac[pop] = float((slopes < 0).mean()) if len(slopes) else 0.
ax.bar([0 - 0.17, 1 - 0.17], [pos_frac["E"], pos_frac["I"]], width=0.34,
       color="#55a868", label="positive slope")
ax.bar([0 + 0.17, 1 + 0.17], [neg_frac["E"], neg_frac["I"]], width=0.34,
       color="#8172b3", label="negative slope")
ax.set_xticks([0, 1]); ax.set_xticklabels(["E units", "I units"])
ax.set_ylabel("fraction of tuned units")
ax.set_title("Both tuning signs emerge (Fig. 3F-lite)")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "tuning.png"), dpi=200)
plt.show()
print("tuned fractions E:", dict(zip(order, np.round(fe, 3))))
print("tuned fractions I:", dict(zip(order, np.round(fi, 3))))

**Time course of selectivity.** The paper's Fig. 3C tracks the fraction of selective
neurons over the trial and finds a processing cascade: offer value signals peak first,
chosen value follows, and choice rises last, mirroring the computational stages of the
decision. We repeat this by regressing single-unit activity on the variables in sliding
400 ms windows, using the paper's standard criterion ($p < 0.05$ and $R^2 \geq 0.005$).
Trials are aligned to offer onset.

In [ ]:
# time-resolved selectivity on decided standard/risky trials, aligned to offer onset
WIN, STEP, TMAX = int(400 / DT), int(100 / DT), int(2000 / DT)
starts = np.arange(-5, TMAX)

def time_resolved_fraction(eps, rules=(0, 1)):
    '''Fraction of units selective for each variable in sliding windows after offer onset.'''
    trs = []
    for ep in eps:
        info = ep["info"]
        if info["rule"] not in rules or info["n_dec"] != 1. or info["choice"] not in (2, 3):
            continue
        t0, _ = offer_window(info)
        trs.append((t0, ep["states"], info))
    fracs = {k: [] for k in order}
    for s in starts:
        Hs, vs = [], {k: [] for k in order}
        for t0, states, info in trs:
            a, b = t0 + s, t0 + s + WIN
            if a < 0 or b > len(states):
                continue
            Hs.append(states[a:b].mean(axis=0))
            ch = 1.0 if info["choice"] == 2 else 0.0
            vs["OVC"].append(info["evc"]); vs["OVE"].append(info["eve"])
            vs["CV"].append(info["evc"] if ch else info["eve"]); vs["CH"].append(ch)
        if len(Hs) < 50:
            for k in order:
                fracs[k].append(np.nan)
            continue
        Hw = np.stack(Hs)
        for k in order:
            x = np.asarray(vs[k])
            sel = 0
            for u in range(Hw.shape[1]):
                r = linregress(x, Hw[:, u])
                if r.pvalue < 0.05 and r.rvalue ** 2 >= 0.005:
                    sel += 1
            fracs[k].append(sel / Hw.shape[1])
    return {k: np.array(v) for k, v in fracs.items()}

fracs_t = time_resolved_fraction(episodes)
t_axis = starts * DT
fig, ax = plt.subplots(figsize=(7.5, 4))
colors = {"OVC": "#4c72b0", "OVE": "#55a868", "CV": "#c44e52", "CH": "#8172b3"}
for k in order:
    ax.plot(t_axis, fracs_t[k] * 100, color=colors[k], lw=1.5, label=pretty[k])
ax.axvspan(-500, 0, color="#fdebcf", zorder=0)
ax.axvspan(0, 1000, color="#dbe9f6", zorder=0)
ax.text(-250, ax.get_ylim()[1] * 0.95, "rule cue", ha="center", fontsize=8, style="italic")
ax.text(500, ax.get_ylim()[1] * 0.95, "offer", ha="center", fontsize=8, style="italic")
ax.set_xlabel("time from offer onset (ms)")
ax.set_ylabel("selective units (%)")
ax.set_title("Selectivity cascade: offer value, then chosen value, then choice (Fig. 3C-lite)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "selectivity_timecourse.png"), dpi=200)
plt.show()

## 7. Low-dimensional population geometry

At the population level the decision is a geometric operation. We quantify two things.

**Dimensionality.** The participation ratio

$$D_{PR} = \frac{(\sum_i \lambda_i)^2}{\sum_i \lambda_i^2}$$

of the covariance eigenvalues of the decision-epoch activity counts how many dimensions the
population actually uses. The paper finds $D_{PR} \approx 2$ for all binary tasks and
$\approx 3$ only for the ternary task: the network recruits exactly one extra dimension for
the third option.

**Geometry.** Projecting each trial's state at the end of the offer epoch onto the first two
PCs reveals a structured cloud: states cluster by the final choice, and position within a
cluster is graded by chosen value. Regressing the PC projections on the task variables (the
paper's principal component regression, its Fig. 4C) shows that PC1 carries chosen value and
PC2 carries the choice, with PC3 recruited only in the ternary task.

In [ ]:
def participation_ratio(X):
    eig = np.linalg.eigvalsh(np.cov((X - X.mean(0)).T))
    eig = eig[eig > 1e-12]
    return float(eig.sum() ** 2 / (eig ** 2).sum())

def decision_states(eps, rule, ms=200.):
    '''States from the last `ms` ms of the offer epoch, concatenated over trials.'''
    out = []
    for ep in eps:
        info = ep["info"]
        if info["rule"] != rule:
            continue
        t0, t1 = offer_window(info, len(ep["states"]))
        n = max(1, int(ms / DT))
        out.append(ep["states"][max(t0, t1 - n):t1])
    return np.concatenate(out, 0) if out else None

prs, prs_e, prs_i = {}, {}, {}
dec_states = {}
for rule in ALL_RULES:
    X = decision_states(episodes, rule)
    if X is None:
        continue
    dec_states[rule] = X
    prs[rule] = participation_ratio(X)
    prs_e[rule] = participation_ratio(X[:, :e_size])
    prs_i[rule] = participation_ratio(X[:, e_size:])

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(prs))
ax.bar(x - 0.25, [prs_e[r] for r in prs], width=0.25, color="#4c72b0", label="E units")
ax.bar(x, [prs_i[r] for r in prs], width=0.25, color="#c44e52", label="I units")
ax.bar(x + 0.25, [prs[r] for r in prs], width=0.25, color="#8c8c8c", label="all units")
for xi, r in zip(x, prs):
    ax.text(xi + 0.25, prs[r] + 0.05, f"{prs[r]:.1f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([RULE_NAMES[r] for r in prs])
ax.set_ylabel("participation ratio")
ax.set_title("Dimensionality of decision-epoch activity (paper Fig. 4A)")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "participation_ratio.png"), dpi=200)
plt.show()
print("PRs (all):", {RULE_NAMES[r]: round(p, 2) for r, p in prs.items()})

In [ ]:
# PCA geometry on risky trials: clusters by choice, graded by chosen value (Fig. 4B)
risky = [ep for ep in episodes
         if ep["info"]["rule"] == 1 and ep["info"]["n_dec"] == 1.
         and ep["info"]["choice"] in (2, 3)]
states_end, ch_end, cv_end = [], [], []
for ep in risky:
    info = ep["info"]
    t0, t1 = offer_window(info, len(ep["states"]))
    states_end.append(ep["states"][max(t0, t1 - int(200 / DT)):t1].mean(axis=0))
    ch_end.append(info["choice"])
    cv_end.append(info["evc"] if info["choice"] == 2 else info["eve"])
S = np.stack(states_end); ch_end = np.asarray(ch_end); cv_end = np.asarray(cv_end)
pca = fit_pca(S, n_components=2)
P = (S - S.mean(0)) @ pca.components.T

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, ch, col, lab in [(axes[0], 2, "#c44e52", "chose C"), (axes[1], 3, "#4c72b0", "chose E")]:
    m = ch_end == ch
    sc = ax.scatter(P[m, 0], P[m, 1], c=cv_end[m], cmap="viridis", s=14, alpha=0.8)
    fig.colorbar(sc, ax=ax, label="chosen value")
    ax.scatter(P[~m, 0], P[~m, 1], c="0.85", s=8, label=f"other trials ({lab.split()[1]} E/C)"
               , zorder=0)
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    ax.set_title(f"trials where the network {lab}")
    ax.legend(fontsize=8, loc="best")
fig.suptitle("End-of-offer population state: separated by choice, graded by value (Fig. 4B)",
             fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "pca_geometry.png"), dpi=200)
plt.show()

# principal component regression per task (Fig. 4C-lite)
def pc_regression(eps, rule):
    '''R^2 of PC1-3 projections vs task variables, over per-trial end-of-offer states.'''
    S, V = [], {"OVC": [], "OVE": [], "CV": [], "CH": []}
    for ep in eps:
        info = ep["info"]
        if info["rule"] != rule or info["n_dec"] != 1.:
            continue
        ch = info["choice"]
        if rule == 3:   # ternary: choices 1/2/3, values eva/evc/eve
            evs = [info["eva"], info["evc"], info["eve"]]
            V["OVC"].append(info["evc"]); V["OVE"].append(info["eve"])
            V["CV"].append(evs[ch - 1]); V["CH"].append(float(ch))
        else:
            if ch not in (2, 3):
                continue
            V["OVC"].append(info["evc"]); V["OVE"].append(info["eve"])
            V["CV"].append(info["evc"] if ch == 2 else info["eve"])
            V["CH"].append(1.0 if ch == 2 else 0.0)
        t0, t1 = offer_window(info, len(ep["states"]))
        S.append(ep["states"][max(t0, t1 - int(200 / DT)):t1].mean(axis=0))
    S = np.stack(S)
    pca = fit_pca(S, n_components=3)
    P = (S - S.mean(0)) @ pca.components.T
    R = np.zeros((3, 4))
    for i in range(3):
        for j, k in enumerate(("OVC", "OVE", "CV", "CH")):
            x = np.asarray(V[k])
            if np.std(x) < 1e-9:
                continue
            R[i, j] = linregress(x, P[:, i]).rvalue ** 2
    return R

R_all = {r: pc_regression(episodes, r) for r in ALL_RULES}
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2), sharey=True)
for ax, r in zip(axes, ALL_RULES):
    im = ax.imshow(R_all[r], vmin=0, vmax=1, cmap="viridis", aspect="auto")
    ax.set_xticks(range(4)); ax.set_xticklabels(["OVC", "OVE", "CV", "CH"], fontsize=9)
    ax.set_yticks(range(3)); ax.set_yticklabels(["PC1", "PC2", "PC3"], fontsize=9)
    ax.set_title(RULE_NAMES[r], fontsize=10)
    for i in range(3):
        for j in range(4):
            ax.text(j, i, f"{R_all[r][i, j]:.2f}", ha="center", va="center",
                    fontsize=8, color="w" if R_all[r][i, j] < 0.6 else "k")
fig.colorbar(im, ax=axes, label="$R^2$", shrink=0.8)
fig.suptitle("What the top PCs encode, per task (paper Fig. 4C, single network)", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "pc_regression.png"), dpi=200)
plt.show()

## 8. Feedforward value computation: where values are computed and stored

The paper's first mechanistic claim is a functional dissociation: value *computation* is
feedforward, value *comparison* is recurrent. The causal test is simple: set
$W_{rec} = 0$ in the trained network. If values are computed by the feedforward pathway, the
lesioned network should still *represent* offer values but be unable to *choose*.

To run this and the later lesion experiments we need one tool the training loop does not
provide: episode collection with lesions. The helper below mirrors
`neuralrnn.rl.collect_episodes` but can clamp a chosen set of units to zero after every
recurrence step, or run with the recurrent weights zeroed (restored afterwards).

In [ ]:
@torch.no_grad()
def collect_episodes_lesioned(agent, envs, n_episodes, *, clamp_units=None,
                              zero_rec=False, device=DEVICE):
    '''Collect episodes with optional lesions (same record format as collect_episodes).

    clamp_units: indices of core units clamped to 0 after every recurrence step.
    zero_rec: run with W_rec = 0 (feedforward isolation); weights are restored after.
    '''
    core = agent.core
    saved = core.h2h.weight.data.clone() if zero_rec else None
    if zero_rec:
        core.h2h.weight.data.zero_()
    try:
        agent.eval()
        episodes = []
        B = envs.num_envs
        obs_np, _ = envs.reset()
        obs = torch.as_tensor(obs_np, dtype=torch.float32, device=device)
        done = torch.ones(B, device=device)
        z = agent.init_state(B, device)
        live = [dict(obs=[], states=[], actions=[], rewards=[], values=[])
                for _ in range(B)]
        while len(episodes) < n_episodes:
            z = agent.recurrence(obs, z * (1.0 - done).unsqueeze(-1))
            if clamp_units is not None:
                z[:, clamp_units] = 0.0
            dist = agent._policy_dist(z)
            action = dist.sample()
            value = agent._value(z)
            action_np = action.cpu().numpy()
            next_obs_np, reward, term, trunc, infos = envs.step(action_np)
            for i in range(B):
                live[i]["obs"].append(obs_np[i])
                live[i]["states"].append(z[i].cpu().numpy())
                live[i]["actions"].append(action_np[i])
                live[i]["rewards"].append(reward[i])
                live[i]["values"].append(value[i].item())
                ep = infos[i].get("episode") if isinstance(infos[i], dict) else None
                if ep is not None and len(episodes) < n_episodes:
                    rec = live[i]
                    episodes.append({
                        "obs": np.asarray(rec["obs"], dtype=np.float32),
                        "states": np.asarray(rec["states"], dtype=np.float32),
                        "actions": np.asarray(rec["actions"], dtype=np.int64),
                        "rewards": np.asarray(rec["rewards"], dtype=np.float32),
                        "values": np.asarray(rec["values"], dtype=np.float32),
                        "info": ep})
                    live[i] = dict(obs=[], states=[], actions=[], rewards=[], values=[])
            obs_np = next_obs_np
            obs = torch.as_tensor(obs_np, dtype=torch.float32, device=device)
            done = torch.as_tensor(np.maximum(term, trunc), dtype=torch.float32,
                                   device=device)
        return episodes
    finally:
        if zero_rec:
            core.h2h.weight.data.copy_(saved)

def performance(eps):
    '''Trial completion (PD) and accuracy on completed trials (PC), both in %.'''
    pd = np.mean([ep["info"]["n_dec"] for ep in eps]) * 100
    decided = [ep["info"]["n_corr"] for ep in eps if ep["info"]["n_dec"] == 1.]
    return {"pd": pd, "pc": np.mean(decided) * 100 if decided else 0.0}

# the causal test on the risky task
envs = SyncVectorEnv([make_env("echochoice", rules=(1,), seed=40_000 + i) for i in range(8)])
eps_intact = collect_episodes_lesioned(agent_mt, envs, 500)
eps_feedforward = collect_episodes_lesioned(agent_mt, envs, 500, zero_rec=True)
envs.close()
perf_intact, perf_ff = performance(eps_intact), performance(eps_feedforward)
print(f"intact network : completion {perf_intact['pd']:.0f}%, accuracy {perf_intact['pc']:.0f}%")
print(f"W_rec = 0      : completion {perf_ff['pd']:.0f}%, accuracy {perf_ff['pc']:.0f}%")

With the recurrent weights removed the network can no longer decide: without recurrent
dynamics there is no fixation holding and no competition, so trials abort. The question is
what the isolated feedforward pathway still *represents*.

**Multiplication from additive synapses.** An expected value multiplies quantity by
probability, but every synapse adds. The paper shows that the feedforward pathway
approximates the product anyway: it builds, over the domain of experienced offers, a
non-linear function of $(q, p)$ that tracks $q \cdot p$ far better than $q + p$. The test
regresses the first two PCs of the lesioned network's offer-epoch activity on either the
multiplicative predictors $(q_C p_C,\; q_E p_E)$ or the additive ones
$(q_C + p_C,\; q_E + p_E)$. (A constructive proof that a ReLU population can approximate
multiplication by tiling the input space with ramp-like basis functions
$r_i = [w_{iq} q + w_{ip} p + b_i]_+$ is given in the paper's STAR Methods; its toy-model
experiment, which shows that the observed rotation of the representation towards value
sum/difference axes requires a mixed-selective input population, is Fig. 5C-D.)

In [ ]:
# PCA of the lesioned (feedforward-only) network's activity, risky task
feats, qc, pc_, qe, pe = [], [], [], [], []
for ep in eps_feedforward:
    info = ep["info"]
    t0, t1 = offer_window(info, len(ep["states"]))
    n = max(1, int(200 / DT))
    seg = ep["states"][max(t0, t1 - n):t1]
    feats.append(seg)
    qc += [info["nc"] / 10.] * len(seg); pc_ += [info["pc"]] * len(seg)
    qe += [info["ne"] / 10.] * len(seg); pe += [info["pe"]] * len(seg)
Hf = np.concatenate(feats, 0)
qc, pc_, qe, pe = map(np.asarray, (qc, pc_, qe, pe))
pca_ff = fit_pca(Hf, n_components=2)
Pf = (Hf - Hf.mean(0)) @ pca_ff.components.T

def r2_multi(y, X):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    ss_res = ((y - X @ beta) ** 2).sum(); ss_tot = ((y - y.mean()) ** 2).sum()
    return float(1 - ss_res / ss_tot)

X_mult = np.stack([qc * pc_, qe * pe], 1)
X_add = np.stack([qc + pc_, qe + pe], 1)
r2_mult = [r2_multi(Pf[:, k], X_mult) for k in range(2)]
r2_add = [r2_multi(Pf[:, k], X_add) for k in range(2)]

fig, ax = plt.subplots(figsize=(5.5, 4))
x = np.arange(2)
ax.bar(x - 0.17, r2_mult, width=0.34, color="#4c72b0",
       label=r"multiplicative model ($q \cdot p$)")
ax.bar(x + 0.17, r2_add, width=0.34, color="#c44e52", label=r"additive model ($q + p$)")
for xi, v in zip(x - 0.17, r2_mult):
    ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)
for xi, v in zip(x + 0.17, r2_add):
    ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(["PC 1", "PC 2"])
ax.set_ylabel("$R^2$")
ax.set_title("Feedforward activity is multiplicative, not additive (Fig. 5B-lite)")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "feedforward_multiplicative.png"), dpi=200)
plt.show()

**Where the values are stored.** The relative values $\rho_X$ are never part of the input.
The paper shows they are physically stored in the *input weights*: across 50 networks
trained with different imposed values of good C, the average input weight from the $q_C$
channel scales linearly with the behaviorally inferred $\hat\rho_C$ ($R^2 = 0.84$, its
Fig. 5E; we cannot repeat that across-network experiment with one network).

What we can inspect in one network is the *wiring structure*. Two analyses from the paper:

1. the correlation matrix of input-weight vectors across offer features (its Fig. 5A):
   quantity and probability channels of the same offer should correlate, because both carry
   that offer's value;
2. a per-neuron selectivity index on the quantity weights,
   $SI_i = (w_{i,q_C} - w_{i,q_E}) / (w_{i,q_C} + w_{i,q_E})$.
   $SI = \pm 1$ means the neuron reads only one offer; $SI = 0$ means balanced input from
   both. The paper finds a dual architecture: specialized peaks near $\pm 1$ *plus* a broad
   mixed population near 0 (its Fig. S9). The mixed population is what rotates the
   feedforward representation onto value sum/difference axes, which makes the downstream
   comparison easy to read out.

In [ ]:
W_in = agent_mt.core.input2h.weight.detach().numpy()   # (units, 16), non-negative
feat_names = ["q_A", "q_B", "q_C", "q_D", "q_E", "p_A", "p_B", "p_C", "p_D", "p_E"]
C_feat = np.corrcoef(W_in[:, 6:16].T)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax = axes[0]
im = ax.imshow(C_feat, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(feat_names, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(feat_names, fontsize=9)
for i in range(10):
    for j in range(10):
        ax.text(j, i, f"{C_feat[i, j]:.1f}", ha="center", va="center", fontsize=6.5,
                color="w" if abs(C_feat[i, j]) > 0.6 else "k")
fig.colorbar(im, ax=ax, label="weight correlation")
ax.set_title("Input weights: quantity and probability\nof the same offer correlate (Fig. 5A-lite)")

ax = axes[1]
wqC, wqE = W_in[:, 8], W_in[:, 10]
SI = (wqC - wqE) / (wqC + wqE + 1e-12)
ax.hist(SI, bins=41, range=(-1, 1), color="#4c72b0")
ax.axvline(0, color="gray", ls=":", lw=1)
ax.set_xlabel(r"selectivity index  $(w_{q_C} - w_{q_E}) / (w_{q_C} + w_{q_E})$")
ax.set_ylabel("units")
ax.set_title("Dual input architecture: specialized + mixed (Fig. S9-lite)")
ax.grid(alpha=0.3, axis="y")
frac_spec = float((np.abs(SI) > 0.8).mean()); frac_mixed = float((np.abs(SI) < 0.2).mean())
ax.text(0.02, 0.95, f"|SI| > 0.8: {frac_spec * 100:.0f}% of units\n"
                    f"|SI| < 0.2: {frac_mixed * 100:.0f}% of units",
        transform=ax.transAxes, va="top", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "input_weights.png"), dpi=200)
plt.show()

## 9. Value comparison in the recurrent circuit: the CRI motif

The second mechanistic claim is about the recurrent circuit: values are compared by a
**competitive recurrent inhibition (CRI)** motif that produces winner-take-all dynamics.
Choice-selective excitatory neurons excite same-choice inhibitory neurons, and those
inhibitory neurons suppress the excitatory neurons selective for the *competing* choice:

```
CHC_E  ──▶  CHC_I  ──▶|  CHE_E          (E: excitatory pools, I: inhibitory pools,
CHE_E  ──▶  CHE_I  ──▶|  CHC_E           ──▶| denotes suppression)
```

We build up the evidence in three steps, following the paper: the decision is visible in the
policy outputs long before the go cue; the winner-take-all structure is confirmed by
regression; and the motif itself is read off the learned recurrent weights, then validated
by causal lesions.

**Step 1: output dynamics.** We replay the recorded hidden states of decided risky trials
through the actor and critic heads and align the traces to the go cue, split by the final
choice. We plot the raw actor *logits* rather than softmax probabilities: before the go cue
the fixation logit towers over the choice logits (that is what holds fixation), so
probabilities would show a flat zero and hide the signal. At the logit level the paper's
Fig. 6A pattern is visible: the fixation output dominates until the go cue while the two
choice outputs diverge progressively during the offer epoch, and the critic tracks chosen
value regardless of which good was chosen.

In [ ]:
# risky-only dataset for the circuit analyses (decided trials)
envs = SyncVectorEnv([make_env("echochoice", rules=(1,), seed=45_000 + i) for i in range(8)])
eps_risky = collect_episodes_lesioned(agent_mt, envs, 600)
envs.close()
eps_rd = [ep for ep in eps_risky
          if ep["info"]["n_dec"] == 1. and ep["info"]["choice"] in (2, 3)]
print(f"{len(eps_rd)} decided risky trials")

# actor logits and critic value from recorded states
W_act = agent_mt.actor.weight.detach().numpy()
b_act = agent_mt.actor.bias.detach().numpy()
scale = agent_mt.config.policy_logit_scale
W_crit = agent_mt.critic.weight.detach().numpy().reshape(-1)
b_crit = float(agent_mt.critic.bias.detach())

def head_outputs(ep):
    H = ep["states"][:, :e_size]
    logits = (H @ W_act.T + b_act) * scale
    value = H @ W_crit + b_crit
    return logits, value

PRE, POST = int(2000 / DT), int(500 / DT)
t_axis = (np.arange(PRE + POST) - PRE) * DT

def aligned(eps, mask_fn):
    '''Align actor logits + critic value to the go cue for a subset of trials.'''
    P, V = [], []
    for ep in eps:
        if not mask_fn(ep):
            continue
        tg = int(ep["info"]["t_go"] / DT)
        logits, val = head_outputs(ep)
        T = len(logits)
        a, b = tg - PRE, tg + POST
        pa = np.full((PRE + POST, 4), np.nan); va = np.full(PRE + POST, np.nan)
        s0, s1 = max(0, a), min(T, b)
        pa[s0 - a:s1 - a] = logits[s0:s1]; va[s0 - a:s1 - a] = val[s0:s1]
        P.append(pa); V.append(va)
    return np.stack(P), np.stack(V)

P_C, V_C = aligned(eps_rd, lambda ep: ep["info"]["choice"] == 2)
P_E, V_E = aligned(eps_rd, lambda ep: ep["info"]["choice"] == 3)

def mean_sem(arr):
    '''Per-bin mean and SEM over trials; bins with <2 valid trials stay NaN.

    Trials differ in length, so early bins are NaN-padded; averaging only where
    data exists avoids the empty-slice warnings.
    '''
    n = np.sum(~np.isnan(arr), axis=0)
    with np.errstate(invalid="ignore"):
        m = np.where(n > 0, np.nansum(arr, axis=0), np.nan) / np.maximum(n, 1)
        var = np.nansum((arr - m) ** 2, axis=0) / np.maximum(n - 1, 1)
    s = np.where(n > 1, np.sqrt(var / np.maximum(n, 1)), np.nan)
    return m, s

panels = [("fixation output", 0), ("choose C output", 2),
          ("choose E output", 3), ("critic value", None)]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), sharex=True)
for ax, (title, idx) in zip(axes, panels):
    if idx is None:
        for V, col, lab in [(V_C, "#c44e52", "chose C"), (V_E, "#4c72b0", "chose E")]:
            m, s = mean_sem(V)
            ax.plot(t_axis, m, color=col, lw=1.5, label=lab)
            ax.fill_between(t_axis, m - s, m + s, color=col, alpha=0.25, lw=0)
        ax.set_ylabel("value (a.u.)")
    else:
        for P, col, lab in [(P_C, "#c44e52", "chose C"), (P_E, "#4c72b0", "chose E")]:
            m, s = mean_sem(P[..., idx])
            ax.plot(t_axis, m, color=col, lw=1.5, label=lab)
            ax.fill_between(t_axis, m - s, m + s, color=col, alpha=0.25, lw=0)
        ax.set_ylabel("output logit")
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_xlabel("time from go cue (ms)")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle("Actor logits and critic value around the go cue, split by final choice "
             "(paper Fig. 6A)", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "output_dynamics.png"), dpi=200)
plt.show()

**Step 2: winner-take-all at the readout.** The WTA claim is that each choice output
carries the value of its own offer *only on trials where that offer is chosen*. The paper
tests this with conjunctive variables: $\mathrm{CV}_C$ equals the value of offer C on trials
where C was chosen and is 0 otherwise, likewise $\mathrm{CV}_E$. Regressing each output
node's offer-epoch activity on these variables should give high $R^2$ on the matching
diagonal and low $R^2$ off-diagonal. The critic, by contrast, should track chosen value
($\mathrm{CV}$) irrespective of choice identity.

In [ ]:
# WTA regression heatmap (paper Fig. 6B, risky task)
acts, evc, eve, chc = [], [], [], []
for ep in eps_rd:
    info = ep["info"]
    t0, t1 = offer_window(info, len(ep["states"]))
    logits, val = head_outputs(ep)
    acts.append(np.concatenate([logits[t0:t1].mean(0), [val[t0:t1].mean()]]))
    evc.append(info["evc"]); eve.append(info["eve"])
    chc.append(1.0 if info["choice"] == 2 else 0.0)
A = np.stack(acts)
evc, eve, chc = np.asarray(evc), np.asarray(eve), np.asarray(chc)
cv = np.where(chc == 1, evc, eve)
cvc = np.where(chc == 1, evc, 0.)
cve = np.where(chc == 1, 0., eve)
V_reg = {"OV C": evc, "OV E": eve, "CV": cv, "CV C": cvc, "CV E": cve}
rows = ["fix", "choose A", "choose C", "choose E", "critic"]

def r2_1d(y, x):
    if np.std(x) < 1e-9 or np.std(y) < 1e-9:
        return 0.0
    return linregress(x, y).rvalue ** 2

R = np.array([[r2_1d(A[:, i], V_reg[c]) for c in V_reg] for i in range(5)])
fig, ax = plt.subplots(figsize=(6.5, 3.8))
im = ax.imshow(R, vmin=0, vmax=1, cmap="viridis", aspect="auto")
ax.set_xticks(range(5)); ax.set_xticklabels(list(V_reg), fontsize=9)
ax.set_yticks(range(5)); ax.set_yticklabels(rows, fontsize=9)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{R[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="w" if R[i, j] < 0.6 else "k")
fig.colorbar(im, label="$R^2$")
ax.set_title("WTA at the readout: each choice output tracks its own\n"
             "chosen value (paper Fig. 6B, risky task)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "wta_regression.png"), dpi=200)
plt.show()

**Step 3: the motif in the weights.** The full $256 \times 256$ recurrent matrix is hard
to read directly. The paper abstracts it into a reduced circuit diagram: classify every unit
by cell type (E/I) and primary selectivity on the risky task (offer value C, offer value E,
chosen value, choice C, choice E, or unselective), group units into pools, and average the
signed weights between pools. Choice selectivity is split by slope sign: a positive slope on
the choice variable means a CHC neuron (more active when C is chosen), negative means CHE.

We classify with the paper's statistical criterion ($p < 0.01$, $R^2 \geq 0.02$ on the
offer-epoch mean rate; chosen-value pools are separated from offer-value pools by whichever
single-variable model wins). The reduced matrix is built from the *effective* signed weights
$|W_{rec}|\,\mathrm{diag}(\pm 1)$.

One caveat before the numbers: the paper averages pools over 20 networks trained to at
least 90% accuracy on every task. Our single network sits below that bar, and its
choice-selective pools on the risky task are small (a handful of units), so the CH rows and
columns of the reduced matrix are estimated from very few units and should be read as
suggestive, not definitive. The value pools (OVC, OVE, CV) are large and their structure is
the robust part of the picture.

In [ ]:
# functional pool classification on the risky task
feats, evc, eve, chc = [], [], [], []
for ep in eps_rd:
    info = ep["info"]
    t0, t1 = offer_window(info, len(ep["states"]))
    feats.append(ep["states"][t0:t1].mean(axis=0))
    evc.append(info["evc"]); eve.append(info["eve"])
    chc.append(1.0 if info["choice"] == 2 else 0.0)
Hp = np.stack(feats); evc, eve, chc = np.asarray(evc), np.asarray(eve), np.asarray(chc)
cv = np.where(chc == 1, evc, eve)
pool_vars = {"OVC": evc, "OVE": eve, "CV": cv, "CH": chc}

labels = []
for u in range(Hp.shape[1]):
    best, best_r2, best_slope = "other", 0.02, 0.0
    for k, x in pool_vars.items():
        r = linregress(x, Hp[:, u])
        if r.pvalue < 0.01 and r.rvalue ** 2 > best_r2:
            best, best_r2, best_slope = k, r.rvalue ** 2, r.slope
    if best == "CH":
        best = "CHC" if best_slope > 0 else "CHE"
    labels.append(best)
labels = np.array(labels, dtype=object)
n_units = Hp.shape[1]
from collections import Counter
print("pool sizes:", dict(Counter(labels)))

W_signed = agent_mt.core.h2h.weight.detach().abs().numpy()
sign = np.ones(n_units); sign[e_size:] = -1.
W_signed = W_signed * sign[None, :]   # effective signed matrix: row = post, col = pre

pools = [f"{v}_{c}" for c in "EI" for v in ("OVC", "OVE", "CV", "CHC", "CHE", "other")]
def pool_idx(name):
    var, ct = name.rsplit("_", 1)
    rng = range(e_size) if ct == "E" else range(e_size, n_units)
    return np.array([u for u in rng if labels[u] == var])

M_red = np.full((len(pools), len(pools)), np.nan)
for i, p_post in enumerate(pools):
    for j, p_pre in enumerate(pools):
        ip, jp = pool_idx(p_post), pool_idx(p_pre)
        if len(ip) and len(jp):
            M_red[i, j] = W_signed[np.ix_(ip, jp)].mean()

fig, ax = plt.subplots(figsize=(6.8, 5.6))
vlim = np.nanmax(np.abs(M_red))
im = ax.imshow(M_red, vmin=-vlim, vmax=vlim, cmap="RdBu_r")
ax.set_xticks(range(len(pools))); ax.set_yticks(range(len(pools)))
ax.set_xticklabels(pools, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(pools, fontsize=9)
ax.set_xlabel("presynaptic pool"); ax.set_ylabel("postsynaptic pool")
for i in range(len(pools)):
    for j in range(len(pools)):
        if not np.isnan(M_red[i, j]):
            ax.text(j, i, f"{M_red[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="w" if abs(M_red[i, j]) > 0.6 * vlim else "k")
fig.colorbar(im, label="mean signed weight")
ax.set_title("Reduced connectivity: the CRI motif (paper Fig. 6C, single network)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cri_matrix.png"), dpi=200)
plt.show()

# the CRI signature, quantitatively
def cell_w(post, pre):
    return M_red[pools.index(post), pools.index(pre)]
print("\nCRI signature (expect: same-choice excitation of I, cross-choice inhibition of E):")
printed = False
for ch, other in (("C", "E"), ("E", "C")):
    if not any(np.isnan([cell_w(f"CH{ch}_I", f"CH{ch}_E"), cell_w(f"CH{other}_E", f"CH{ch}_I")])):
        print(f"  CH{ch}_E -> CH{ch}_I: {cell_w(f'CH{ch}_I', f'CH{ch}_E'):+.3f}   "
              f"CH{ch}_I -> CH{other}_E: {cell_w(f'CH{other}_E', f'CH{ch}_I'):+.3f}")
        printed = True
if not printed:
    print("  choice pools too small in this single network for a reliable estimate;")
    print("  the value-pool structure above and the lesions below carry the evidence.")

**Causal validation by lesions.** Structure alone is not mechanism. We clamp each
functional pool to zero in turn and re-measure performance on the risky task. The paper
(Fig. S13) finds that choice accuracy depends on a distributed set of pools, and that
silencing the *inhibitory* choice pools or the excitatory chosen-value pool is at least as
damaging as silencing the excitatory choice pools: both signs of the CRI motif carry the
computation.

A second lesion tests the negative-tuning story from section 6. Negative tuning slopes
cannot come from the (excitatory) inputs; they are created by the CRI's targeted inhibition.
If that inversion is part of the mechanism, silencing all units with significantly negative
tuning for offer value or chosen value should hurt decisions across all tasks
(paper Fig. S12).

In [ ]:
# baseline on all tasks (for the negative-tuning lesion)
perf_base = {}
envs = SyncVectorEnv([make_env("echochoice", seed=60_000 + i) for i in range(8)])
eps_base = collect_episodes_lesioned(agent_mt, envs, 800)
envs.close()
for rule in ALL_RULES:
    sub = [ep for ep in eps_base if ep["info"]["rule"] == rule]
    perf_base[rule] = performance(sub)
print("baseline per rule:", {RULE_NAMES[r]: {k: round(v) for k, v in p.items()}
                             for r, p in perf_base.items()})

# --- pool ablations on the risky task (paper Fig. S13-lite) ---
envs = SyncVectorEnv([make_env("echochoice", rules=(1,), seed=50_000 + i) for i in range(8)])
eps_intact_r = collect_episodes_lesioned(agent_mt, envs, 300)
perf_r_intact = performance(eps_intact_r)
abl_results = {}
for p in pools:
    if p.endswith("_other") or p.startswith("other"):
        continue
    idx = pool_idx(p)
    if len(idx) == 0:
        continue
    perf_l = performance(collect_episodes_lesioned(agent_mt, envs, 300, clamp_units=idx))
    abl_results[p] = (len(idx), perf_l)
    print(f"clamp {p:8s} (n={len(idx):3d}): completion {perf_l['pd']:5.1f}%  "
          f"accuracy {perf_l['pc']:5.1f}%")
envs.close()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
names = list(abl_results)
x = np.arange(len(names))
drops_pd = [perf_r_intact["pd"] - abl_results[n][1]["pd"] for n in names]
drops_pc = [perf_r_intact["pc"] - abl_results[n][1]["pc"] for n in names]
axes[0].bar(x, drops_pd, color="#4c72b0")
axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=45, ha="right", fontsize=9)
axes[0].set_ylabel("drop in trial completion (pp)")
axes[0].set_title("Pool ablation: completion (risky)")
axes[0].grid(alpha=0.3, axis="y")
axes[1].bar(x, drops_pc, color="#c44e52")
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=45, ha="right", fontsize=9)
axes[1].set_ylabel("drop in accuracy (pp)")
axes[1].set_title(f"Pool ablation: accuracy (intact {perf_r_intact['pc']:.0f}%)")
axes[1].grid(alpha=0.3, axis="y")
fig.suptitle("Functional ablation of pools (paper Fig. S13, single network)", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "pool_ablation.png"), dpi=200)
plt.show()

# --- negative-tuning lesion across all tasks (paper Fig. S12-lite) ---
neg_units = []
for u in range(n_units):
    for k in ("OVC", "OVE", "CV"):
        r = linregress(pool_vars[k] if k != "CV" else cv, Hp[:, u])
        if r.pvalue < 0.05 and r.slope < 0:
            neg_units.append(u)
            break
neg_units = np.array(neg_units)
print(f"\nnegatively tuned units (OV or CV, p<0.05): {len(neg_units)}")
envs = SyncVectorEnv([make_env("echochoice", seed=70_000 + i) for i in range(8)])
eps_neg = collect_episodes_lesioned(agent_mt, envs, 800, clamp_units=neg_units)
envs.close()

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(ALL_RULES))
drop = [perf_base[r]["pc"] - performance([ep for ep in eps_neg
                                          if ep["info"]["rule"] == r])["pc"]
        for r in ALL_RULES]
ax.bar(x, drop, color="#8172b3")
ax.set_xticks(x); ax.set_xticklabels([RULE_NAMES[r] for r in ALL_RULES])
ax.set_ylabel("drop in accuracy (pp)")
ax.set_title(f"Silencing negatively tuned units impairs all tasks (Fig. S12-lite, "
             f"{len(neg_units)} units clamped)")
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "negative_tuning_lesion.png"), dpi=200)
plt.show()

## 10. Compositionality: one circuit, five tasks

The last result concerns multitasking. A single network trained on all five tasks could, in
principle, either build five separate subnetworks or share one computational core. The paper
finds a compositional middle ground: a shared core for the common computation (compute
values, compare them) plus small specialized modules for the demands that differ (a third
option in the ternary task, working memory in the sequential task).

We look at three pieces of evidence, all from the paper's Fig. 7:

1. **Rule cues become redundant for similar tasks.** The correlation structure of the rule
   input weights and the principal angles between the task-specific activity subspaces
   (during the rule-cue epoch) show that standard, risky, ternary and bundle tasks share
   their geometry, while the sequential task occupies a nearly orthogonal subspace: its
   working-memory demand genuinely needs different dynamics.
2. **Neurons cluster into shared and specialized modules.** For each unit we compute the
   variance of its activity within each task, normalize by the maximum across tasks, and
   k-means cluster the resulting 5-dimensional profiles (E and I separately; k chosen by
   silhouette score). The paper consistently finds a shared cluster plus ternary- and
   sequential-specialized clusters.
3. **Lesions of the clusters dissociate tasks.** Silencing the sequential cluster should
   hurt only the sequential task, and so on: the modules are causal, not just descriptive.

The fourth panel of that analysis is the fractional task-variance difference,
$(TV_1 - TV_2)/(TV_1 + TV_2)$ per neuron for a pair of tasks: a sharp peak at 0 with
shoulders (inclusive relationship) for standard vs bundle, a bimodal distribution with peaks
at $\pm 1$ (disjoint relationship) for ternary vs sequential.

In [ ]:
# (1) rule input weights and rule-cue subspaces
C_rule = np.corrcoef(W_in[:, :5].T)
rule_labels = [RULE_NAMES[r] for r in ALL_RULES]

def rulecue_states(eps, rule, ms=200.):
    n = max(1, int(ms / DT))
    out = []
    for ep in eps:
        info = ep["info"]
        if info["rule"] != rule:
            continue
        t1 = int((info["t_fix"] + info["t_rule"]) / DT)
        t0 = max(int(info["t_fix"] / DT), t1 - n)
        if t1 > t0:
            out.append(ep["states"][t0:t1])
    return np.concatenate(out, 0) if out else None

subspaces = {}
for r in ALL_RULES:
    X = rulecue_states(eps_base, r)
    n = max(1, int(round(participation_ratio(X))))
    subspaces[r] = fit_pca(X, n_components=n).components

PA = np.full((5, 5), np.nan)
for i, r1 in enumerate(ALL_RULES):
    for j, r2 in enumerate(ALL_RULES):
        s = np.linalg.svd(subspaces[r1] @ subspaces[r2].T, compute_uv=False)
        PA[i, j] = np.degrees(np.arccos(np.clip(s.mean(), -1, 1)))  # mean principal angle

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, Mx, title, vlim in [(axes[0], C_rule, "rule input weight correlations (Fig. 7A)", 1),
                            (axes[1], PA, "principal angles between rule-cue subspaces (deg, "
                             "Fig. 7B)", 90)]:
    im = ax.imshow(Mx, vmin=0, vmax=vlim, cmap="magma" if vlim == 90 else "viridis")
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels(rule_labels, rotation=30, ha="right", fontsize=9)
    ax.set_yticklabels(rule_labels, fontsize=9)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{Mx[i, j]:.2f}" if vlim == 1 else f"{Mx[i, j]:.0f}",
                    ha="center", va="center", fontsize=8,
                    color="w" if Mx[i, j] < 0.6 * vlim else "k")
    fig.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "rule_subspaces.png"), dpi=200)
plt.show()

In [ ]:
# (2) task-variance profiles and clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

tv = np.zeros((n_units, 5)); counts = np.zeros(5)
for ep in eps_base:
    info = ep["info"]; r = int(info["rule"])
    t0, t1 = offer_window(info, len(ep["states"]))
    if t1 - t0 < 2:
        continue
    tv[:, r] += ep["states"][t0:t1].var(axis=0)
    counts[r] += 1
tv /= np.maximum(counts[None, :], 1)
tvn = tv / np.maximum(tv.max(axis=1, keepdims=True), 1e-12)

def cluster_profiles(prof, ks=range(2, 9)):
    best = max(((silhouette_score(prof, KMeans(n_clusters=k, n_init=10,
                random_state=0).fit_predict(prof)), k) for k in ks))
    s, k = best
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(prof)
    return k, s, lab

kE, sE, labE = cluster_profiles(tvn[:e_size])
kI, sI, labI = cluster_profiles(tvn[e_size:])
print(f"E: k={kE} (silhouette {sE:.2f}), sizes {np.bincount(labE)}")
print(f"I: k={kI} (silhouette {sI:.2f}), sizes {np.bincount(labI)}")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, lab, prof, k, pop in [(axes[0], labE, tvn[:e_size], kE, "excitatory"),
                              (axes[1], labI, tvn[e_size:], kI, "inhibitory")]:
    order_idx = np.lexsort((-prof[:, 2], lab))  # group by cluster, sequential var within
    im = ax.imshow(prof[order_idx].T, aspect="auto", vmin=0, vmax=1, cmap="viridis")
    ax.set_yticks(range(5)); ax.set_yticklabels(rule_labels, fontsize=9)
    bounds = np.cumsum([np.sum(lab == c) for c in range(k)])
    for b in bounds[:-1]:
        ax.axvline(b - 0.5, color="r", lw=1)
    for c in range(k):
        ctr = prof[lab == c].mean(0)
        top = rule_labels[int(np.argmax(ctr))]
        mid = (bounds[c - 1] if c else 0) / 2 + bounds[c] / 2 if c else bounds[0] / 2
        ax.text(mid, -0.7, f"cl{c}: {top}", ha="center", fontsize=8, color="r")
    ax.set_xlabel("units (sorted by cluster)")
    ax.set_title(f"{pop} units, k={k}", fontsize=10)
fig.colorbar(im, ax=axes, label="normalized task variance", shrink=0.8)
fig.suptitle("Task-variance profiles cluster into shared and specialized modules "
             "(paper Fig. 7C)", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "task_variance_clusters.png"), dpi=200)
plt.show()

# (3) cluster lesions across tasks (paper Fig. S14-lite)
cluster_defs = {f"E cluster {c}": np.where(labE == c)[0] for c in range(kE)}
cluster_defs.update({f"I cluster {c}": np.where(labI == c)[0] + e_size for c in range(kI)})
lesion_rows = []
for name, idx in cluster_defs.items():
    if len(idx) == 0:
        continue
    eps_l = collect_episodes_lesioned(agent_mt,
            SyncVectorEnv([make_env("echochoice", seed=80_000 + i) for i in range(8)]),
            800, clamp_units=idx)
    row = [perf_base[r]["pc"] - performance([ep for ep in eps_l
                                             if ep["info"]["rule"] == r])["pc"]
           for r in ALL_RULES]
    lesion_rows.append((name, row))
    print(f"clamp {name} (n={len(idx)}): accuracy drop per task "
          f"{[f'{v:.0f}' for v in row]}")

fig, ax = plt.subplots(figsize=(8, 4.2))
M_les = np.array([r for _, r in lesion_rows])
im = ax.imshow(M_les, cmap="Reds", vmin=0, aspect="auto")
ax.set_xticks(range(5)); ax.set_xticklabels(rule_labels, fontsize=9)
ax.set_yticks(range(len(lesion_rows)))
ax.set_yticklabels([n for n, _ in lesion_rows], fontsize=9)
for i in range(len(lesion_rows)):
    for j in range(5):
        ax.text(j, i, f"{M_les[i, j]:.0f}", ha="center", va="center", fontsize=8,
                color="w" if M_les[i, j] > 0.6 * M_les.max() else "k")
fig.colorbar(im, label="accuracy drop (pp)")
ax.set_title("Lesioning a module selectively impairs its task (paper Fig. S14-lite)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "cluster_lesions.png"), dpi=200)
plt.show()

# (4) fractional task-variance differences (paper Fig. 7D)
def fvd(a, b):
    return (tvn[:, a] - tvn[:, b]) / (tvn[:, a] + tvn[:, b] + 1e-12)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (a, b) in zip(axes, [(0, 4), (3, 2)]):
    ax.hist(fvd(a, b), bins=41, range=(-1, 1), color="#4c72b0")
    ax.axvline(0, color="gray", ls=":", lw=1)
    ax.set_xlabel(rf"$(TV_{{{rule_labels[a]}}} - TV_{{{rule_labels[b]}}}) / "
                  rf"(TV_{{{rule_labels[a]}}} + TV_{{{rule_labels[b]}}})$")
    ax.set_title(f"{rule_labels[a]} vs {rule_labels[b]}", fontsize=10)
    ax.grid(alpha=0.3, axis="y")
axes[0].set_ylabel("units")
fig.suptitle("Inclusive vs disjoint task relationships (paper Fig. 7D)", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "task_variance_differences.png"), dpi=200)
plt.show()

## Summary

One E-I recurrent network, trained with PPO purely from reward feedback, reproduces the
paper's two-stage circuit framework for economic choice:

* **Behavior.** Choices follow a steep value-based psychometric function, and the fitted
  relative value of good C recovers the ground truth ($\hat\rho_C \approx 2$) that the
  network could only learn through rewards. Reaction times do not grow with decision
  difficulty, a behavioral hint that the decision is made during the offer epoch.
* **Single units.** The OFC cell types (offer value, chosen value, choice) emerge in both
  the excitatory and the inhibitory population, with a temporal cascade from offer value to
  chosen value to choice, and with both positive and negative tuning slopes.
* **Population geometry.** Decision-epoch activity is low-dimensional (participation ratio
  around 2 for binary tasks, higher for ternary) and organized by choice and chosen value.
* **Feedforward value computation.** Removing the recurrent weights abolishes decisions but
  leaves a feedforward activity pattern that is better explained by the *product* of
  quantity and probability than by their sum; the input weights show the dual specialized +
  mixed architecture that rotates the representation onto sum/difference axes.
* **Recurrent value comparison.** The policy outputs commit to a choice before the go cue,
  each choice output carries its own offer's value only when chosen (WTA), and the reduced
  pool-to-pool connectivity shows the CRI motif. Silencing inhibitory choice pools or
  negatively tuned units degrades decisions causally.
* **Compositionality.** The four simultaneous-offer tasks share their rule-cue geometry
  while the sequential task occupies its own subspace; units cluster into a shared module
  and task-specialized modules whose lesions selectively impair the matching tasks.

**What we did not reproduce** (all require many independently trained networks, not one):
the across-network distributions of behavioral parameters (Fig. 2C-E), the linear law
between input weights and imposed good values over 50 networks (Fig. 5E), the toy-model
rotation experiment (Fig. 5C-D), the Procrustes alignment of 20 networks (Fig. 7E), the
zero-shot transfer matrix (Fig. S15), the curriculum speedup (Fig. 7F), and the
novel-offer generalization experiment (Fig. 8). The framework supports all of them; they are
loops of what this notebook does once.

**Where to go next.** The RL layer is generic: swap the environment (`make_env` also serves
`neurogym:<Task>` and `gym:<id>`), the algorithm (register a new loss with
`register_rl_loss`), or the core (`ActorCriticConfig.core_config` accepts any registered
model type). Notebook 19 uses the same machinery for a critic-only value RNN, and
`s1_plumetracknets` for a continuous-action navigation task.